<a href="https://colab.research.google.com/github/Mat9408/Atividades_CursoAnalistaDeDados_EBAC/blob/main/M%C3%B3dulo_39_Tarefa_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Atividade 1: Extrair dados do site da B3 através de uma API

In [ ]:
!pip install boto3

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 72.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 9.9 MB/s eta 0:00:00


In [ ]:
import json
from datetime import datetime

import requests

#-- setup

URL= 'https://www2.cetip.com.br/ConsultarTaxaDi/ConsultarTaxaDICetip.aspx'

#-- extract

try:
  response = requests.get(URL)
  response.raise_for_status()
except Exception as exc:
    raise exc
else:
  data = json.loads(response.text)
  print(f'1 - {data}')

#-- transform

data['taxa'] = data['taxa'].replace(',', '.')
data['indice'] = data['indice'].replace('.', '').replace(',','.')

data['dataTaxa'] = datetime.strptime(data['dataTaxa'], '%d/%m/%Y').strftime('%Y-%m-%d')
data['dataIndice'] = datetime.strptime(data['dataIndice'], '%d/%m/%Y').strftime('%Y-%m-%d')

data['dataReferencia'] = datetime.now().strftime('%Y-%m-%d')

data_csv = ','.join([v for v in data.values()])

print(f'2 - {data}')
print(f'3 - {data_csv}')



1 - {'taxa': '13,65', 'dataTaxa': '09/06/2023', 'indice': '40.040,19', 'dataIndice': '12/06/2023'}
2 - {'taxa': '13.65', 'dataTaxa': '2023-06-09', 'indice': '40040.19', 'dataIndice': '2023-06-12', 'dataReferencia': '2023-06-12'}
3 - 13.65,2023-06-09,40040.19,2023-06-12,2023-06-12


In [ ]:
import boto3

client = boto3.client('s3')
client.upload_file(Filename='<nome-do-arquivo>', Bucket='<nome-do-bucket>', Key='<nome-do-objeto>')

client = boto3.client('athena')
client.start_query_execution(
    QueryString='SELECT * FROM <nome-da-tabela> LIMIT 10',
    ResultConfiguration={'OutputLocation': 's3://<nome-do-bucket-de-resultados>/'}
)

FileNotFoundError: ignored

  - AWS Lambda - Bucket **bronze**

In [ ]:
import json
import logging
from datetime import datetime

import boto3
import urllib3
from botocore.exceptions import ClientError

def lambda_handler(event,context) -> bool:

  #-- setup

  URL = 'https://www2.cetip.com.br/ConsultarTaxaDi/ConsultarTaxaDICetip.aspx'
  BRONZE_BUCKET = 'bucket-mt9408-modulo39-bronze'

  client = boto3.client('s3')

  date = datetime.now().strftime('%Y-%m-%d')
  filename_json = f'stock-exchange-{date}.json'

  #-- extract

  try:
    http = urllib3.PoolManager()
    response = http.request(url=URL, method='get')
  except Exception as exc:
    raise exc
  else:
    data = json.loads(response.data.decode())
    logging.info(msg=data)

  #-- transform

  ...

  #-- load

  try:
      with open(f'/tmp/{filename_json}', mode='w', encoding='utf8') as fp:
          json.dump(data,fp)
      client.upload_file(Filename=f'/tmp/{filename_json}', Bucket=BRONZE_BUCKET, Key=filename_json)
  except ClientError as exc:
    raise exc

  return json.dumps(dict(status=True))

  - AWS Lambda para bucket **silver**

In [ ]:
import json
from datetime import datetime

import boto3
from botocore.exceptions import ClientError

def lambda_handler(event, context) -> bool:

  #-- setup

  BRONZE_BUCKET = 'bucket-mt9408-modulo39-bronze'
  SILVER_BUCKET = 'bucket-mt9408-modulo39-silver'

  client = boto3.client('s3')

  date = datetime.now().strftime('%Y-%m-%d')
  filename_csv = f'stock-exchange-{date}.csv'
  filename_json = f'stock-exchange-{date}.json'

  #-- extract

  client.download_file(BRONZE_BUCKET, filename_json, f'/tmp/{filename_json}')

  with open(f'/tmp/{filename_json}', mode='r', encoding='utf8') as fp:
    data = json.load(fp)

  #-- transorm

  data['taxa'] = data['taxa'].replace(',', '.')
  data['indice'] = data['indice'].replace('.', '').replace(',','.')

  data['dataTaxa'] = datetime.strptime(data['dataTaxa'], '%d/%m/%Y').strftime('%Y-%m-%d')
  data['dataIndice'] = datetime.strptime(data['dataIndice'], '%d/%m/%Y').strftime('%Y-%m-%d')

  #-- load

  try:
        with open(f'/tmp/{filename_csv}', mode='w', encoding='utf8') as fp:
          fp.write(','.join([v for v in data.values()]))
        client.upload_file(Filename=f'/tmp/{filename_csv}', Bucket=SILVER_BUCKET, Key=f'data_referencia={date}/{filename_csv}')
  except ClientError as exc:
    raise exc

  return json.dumps(dict(status=True))

  - AWS Lambda para tabela

In [ ]:
import json
from datetime import datetime

import boto3
from botocore.exceptions import ClientError

def lambda_handler(event, context) -> bool:

  #-- setup

  SILVER_BUCKET = 'bucket-mt9408-modulo39-silver'

  query = f"""
  CREATE EXTERNAL TABLE IF NOT EXISTS cdi (
    taxa double,
    data_taxa string,
    indice double,
    data_indice string
  )
  PARTITIONED BY (
    data_referencia string
  )
  ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
  WITH SERDEPROPERTIES ('separatorChar'=',')
  LOCATION 's3://{SILVER_BUCKET}/'
  """

  client = boto3.client('athena')

  # -- create

  try:
    client.start_query_execution(
        QueryString=query,
        ResultConfiguration={
            'OutputLocation': 's3://mt9408-modulo38-athena-results/'
        }
    )
  except ClientError as exc:
    raise exc

  #--update

  try:
    client.start_query_execution(
        QueryString='MSCK REPAIR TABLE cdi',
        ResultConfiguration={
            'OutputLocation': 's3://mt9408-modulo38-athena-results/'
        }
    )
  except ClientError as exc:
    raise exc

  return json.dumps(dict(status=True))